<div style="font-size: 2.2em; font-weight: bold; line-height: 1.3;">Final Exam</div>

<div style="font-size: 1.5em; font-weight: bold; line-height: 1.5;">BUSN 41210: Financial Analytics</div>

<div style="font-size: 1.5em; font-weight: bold; line-height: 1.5;">Due: 5:00 pm on Sunday, Oct. 4th, 2026</div>

**Use of AI tools.** You may use any AI tool (ChatGPT, Claude, Copilot or others) for any part of this exam: code, explanations, writing. You are responsible for every answer you submit, exactly as if you had written it yourself. Every number must come from a cell in this notebook, and an answer you cannot explain or reproduce is your error, not the tool's.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



---
# Problem 1: Trees and Ensembles (20 points)

Lecture 8 built a single tree, then pruned it, then averaged many of them. This
problem walks that ladder on a small dataset where you can actually see the tree.

`Social_Network_Ads.csv` records 400 people shown an advertisement: their
`Gender`, `Age` and `EstimatedSalary`, and whether they `Purchased` (1) or not.
Use **5-fold stratified cross-validation with `random_state=7034`** for every
accuracy you report, and pass `random_state=7034` to every tree, forest and boosting model, so
the numbers are comparable to each other.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.inspection import permutation_importance

_DATA_DIR = '/classes/41210_MiF_fall2026/Data/'
ads = pd.read_csv(_DATA_DIR + 'Social_Network_Ads.csv')
# ads = pd.read_csv('Social_Network_Ads.csv')
ads['Gender'] = (ads.Gender == 'Male').astype(int)

Xa = ads[['Gender', 'Age', 'EstimatedSalary']]
ya = ads['Purchased']

cv5 = StratifiedKFold(5, shuffle=True, random_state=7034)
def cv_acc(model):
    """5-fold cross-validated accuracy."""
    return cross_val_score(model, Xa, ya, cv=cv5, scoring='accuracy').mean()

print(f"{len(ads)} people, {ya.mean():.1%} of them purchased")

**1.1.** Before any model: what accuracy does the rule *"nobody purchases"* achieve? Report it, and treat it as the number every model below has to beat.

Now grow an **unpruned** classification tree and report its 5-fold accuracy. Then sweep `max_leaf_nodes` over $\{2, 3, 4, 6, 8, 12\}$ and report the accuracy at each. Which size does cross-validation prefer (if two sizes tie, take the smaller), and what happens on either side of it?

In [ ]:
# Your codes here.


**Answer:** 

**1.2.** Plot the tree you chose in 1.1 (`plot_tree` with `feature_names=Xa.columns`), and then **say what it does in one or two English sentences** — the kind you could put in front of someone with no statistics.

In [ ]:
# Your codes here.


**Answer:** 

**1.3.** Fit a random forest (300 trees) and a gradient boosted model (100 rounds), both with `random_state=7034`, and report their 5-fold accuracies.

Do they beat the tree from 1.1? Report what you find **whichever way it comes out**, and explain in two or three sentences why that might be, given the size and shape of this dataset.

In [ ]:
# Your codes here.


**Answer:** 

**1.4.** Split the data 70/30 (`random_state=7034`, `stratify=ya`), fit a 300-tree forest on the training set, and compute two importance rankings: `feature_importances_` (mean decrease in impurity, computed **in sample**) and `permutation_importance` on the **held-out** 30%.

Put them side by side. Do the two rankings agree on which variable matters most? Where they disagree, explain why — and say which of the two you would show to someone making a decision.

In [ ]:
# Your codes here.


**Answer:** 

**1.5** (Open-Ended) Each row here is a *person*, and the rows are unrelated to each other. Suppose instead each row were a *month* of one stock's data, and you were predicting whether next month's return is positive.

Cross-validation with `shuffle=True` — which you used throughout this problem — would then be **wrong**. Explain precisely what goes wrong, what you would do instead, and which way the error would push your reported accuracy.

In [ ]:
# Your codes here (if any).


**Answer:** 

---
# Problem 2: Training on Your Own Output (20 points)

A risk desk estimates the daily volatility of its portfolio from a library of past daily returns
and reports the one-day 1% Value-at-Risk. Under a fitted normal distribution $N(0, \hat\sigma^2)$
the 1% quantile is $-2.326\,\hat\sigma$, so the desk reports $\text{VaR} = 2.326\,\hat\sigma$: under
this model, estimating VaR *is* estimating $\hat\sigma$. In 2016 the desk decided that the library
should be refreshed every night so that it always reflects the newest model. Its pipeline manual:

> *Every night the pipeline **(1)** fits $\hat\sigma^2$ to the library with the usual unbiased sample
> variance, $\hat\sigma^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar x)^2$, and sets the mean of the
> fitted normal to zero; **(2)** draws 500 fresh scenarios from $N(0, \hat\sigma^2)$; **(3)** replaces
> the library with tonight's 500 scenarios, so the library keeps its size, because the newest model is the best model and
> yesterday's data came from a worse one; and **(4)** reports $\text{VaR} = 2.326\,\hat\sigma$.*

On the first night the library held 500 real daily returns. The pipeline then ran every night for
ten years, 2,500 refits in all, and the reported VaR fell from 2.8% of NAV to 0.24%, while the
volatility of the desk's markets had not changed. The desk was praised for de-risking. The Chief Risk Officer, who does not
believe in free lunches, wants to know what happened.

**Rules.** Work in units of the true volatility: $\sigma = 1$, so the *correct* VaR is $2.326$ on
every night and nothing about the market changes. Set `SEED` to the last four digits of your student ID; your numbers will differ from your classmates', and that is the point. A "desk" is one run of the
pipeline; every desk starts on night 1 from a library whose sample variance is exactly 1, and when a
part needs the real days themselves, take them to be 500 draws from $N(0, 1)$. Every number you
report must come from a cell in this notebook.


In [ ]:
from scipy.stats import norm, kurtosis
import time

_DATA_DIR = '/classes/41210_MiF_fall2026/Data/'

SEED   = 0             # <-- the last four digits of your student ID
N_SCEN = 500           # scenarios drawn per night
NIGHTS = 2500          # ten trading years
Z01    = norm.ppf(0.99)   # 2.326: the 1% one-day VaR of N(0, 1), in units of sigma
print(f"correct VaR, every night, in units of sigma: {Z01:.3f}")


**2.1.** (2 points) **Commit before you compute.** Answer in the markdown cell below *before*
writing any code. The estimator in step (1) is unbiased, so each night's estimate is, on average,
equal to the previous night's. Does that mean the reported VaR should stay near 2.326 for ten
years? Say what you expect to see on night 2,500 and why, in two or three sentences. No marks depend
on being right; **2.3** asks you to reconcile your answer with what you find.


**Answer:** 


**2.2.** (6 points) **One desk, then a thousand.** Simulate one desk for 2,500 nights exactly as
the manual describes: fit $\hat\sigma^2$ with `var(ddof=1)`, draw 500 scenarios from
$N(0, \hat\sigma^2)$, replace the library. Plot $\log\hat\sigma^2$ against the night and report
$\hat\sigma^2$ and the reported VaR on night 2,500.

Then run a thousand desks (a loop over nights that draws a $1000 \times 500$ matrix each night takes
seconds). For night 2,500 report the mean, median, 5th and 95th percentiles and maximum of
$\hat\sigma^2$, and the fraction of desks reporting a VaR below one-tenth of the truth. Where does
your one desk sit in that distribution? Is the story's fall from 2.8% to 0.24% typical?


In [ ]:
# Your codes here.


**Answer:** 


**2.3.** (6 points) **Unbiased every night, wrong in the end.** Verify in a cell that
$E[\hat\sigma^2_{t+1} \mid \hat\sigma^2_t] = \hat\sigma^2_t$: start many desks at the same
$\hat\sigma^2_t$, take one step of the pipeline, and average. So the mathematically expected value
of $\hat\sigma^2$ on night 2,500 is exactly 1. Then measure the *average nightly change in
$\log\hat\sigma^2$* for $n = 50$, $500$ and $5{,}000$ scenarios per night (a few hundred desks and a
few hundred nights are enough) and propose the formula relating it to $n$; it is simple. Plot the
histogram of $\log\hat\sigma^2$ across your thousand desks on night 2,500.

In one paragraph, reconcile three numbers that are all correct: the expectation is exactly 1, your
median is far below 1, and your thousand-desk mean is neither (what share of the sum of your
thousand estimates is held by the ten largest?). Why does "unbiased at every step" not protect a
pipeline that trains on its own output?


In [ ]:
# Your codes here.


**Answer:** 


**2.4.** (6 points) **What synthetic data can and cannot carry.** Two experiments.

(a) Keep the 500 real days in the library permanently, and add each night's 500 scenarios to them
instead of replacing them, so the library holds 1,000 returns, half real and half synthetic. Report the median and the 5th and 95th percentiles of $\hat\sigma^2$ on
night 2,500 across a thousand desks.

(b) Seed the library with real returns instead. `dj30.csv` is a daily panel of Dow stocks, one row
per stock per date, and the daily market return `MrkRet` is repeated on every row for that date;
reduce it to one value per date (for example `dj.groupby('date').MrkRet.first()`), which gives 1,511
trading days for 2016–2021 including March 2020. Report the sample volatility and the excess
kurtosis of the real returns. Then compare two things *on night 1, before the pipeline has drawn a
single scenario from its own output*: the empirical 1% VaR (the first percentile of the real
returns, sign flipped) against the VaR the pipeline reports, and the excess kurtosis of the real
returns against that of the 500 night-1 scenarios.

Then answer in one paragraph. What does a synthetic sample contain that the real sample did not,
and what does it lack? Which of the two experiments shows a *loss of information* and which shows
an *accumulation of noise*? What does this imply for any pipeline, a risk model, a return forecast
or a language model, that is retrained on data produced by its own previous version, and what is the
only thing that stops it?


In [ ]:
# Your codes here.


**Answer:** 


---
# Problem 3: Research Project — Cross-country asset return prediction (60 points)

This is an open-ended research project rather than a problem set. You are given a research panel of monthly asset returns across several countries and asset classes, with asset-level characteristics and macroeconomic data, and a question. How you answer it — which features you build, which models you fit, how you evaluate them — is your design, and the write-up must defend it. There is no answer key. A negative result, honestly established, earns full credit; an unexplained positive one does not.

**The question.** The empirical asset-pricing literature finds that a small set of asset-level characteristics — variables that describe each asset's own recent behaviour, pricing and risk, and that differ across assets at a point in time — predict returns across many markets and asset classes, and systematic funds trade on exactly these cross-sectional signals. The panel you are given carries five such characteristics per asset, with their names hidden, together with global and country-level macro variables. What a desk that trades across countries and asset classes wants to know is:

1. How much of next month's excess return is forecastable from lagged characteristics?
2. Does macroeconomic information — global or country-level — add anything to the characteristics?
3. Does a forecast-based portfolio beat the simple rules (equal weight, risk parity, time-series momentum) out of sample, after costs?

Any method is allowed: OLS, ridge, lasso, PCR, KNN, trees, random forests, boosting, neural networks, or anything else you want to try; static, expanding-window or rolling-window estimation; cross-validation or information criteria for tuning. The choice is yours; the discipline of the evaluation is what is graded.


## Data

All files are in `_DATA_DIR`. Dates are month-ends, January 2000 to December 2024. Returns are monthly **excess** returns in decimals. Asset, asset-class, country and variable names are anonymized.

| file | grain | columns |
|---|---|---|
| `asset_panel.csv` | asset × month (50 × 300) | `date, asset_id, asset_class, country, excess_return, x1 ... x5` |
| `asset_returns_wide.csv` | month × asset | the same returns, one column per asset |
| `asset_info.csv` | asset | `asset_class` (A, B, C, D) and `country` of each asset |
| `macro_global.csv` | month | `x6 ... x9`, global variables |
| `macro_country.csv` | country × month, 12 countries | `x10 ... x16`, curated country-level variables |
| `macro_country_extended.csv` | country × month, 12 countries | `x17 ... x164`, a wide, uncurated set of further country-level variables that overlaps the curated file (a few of its columns duplicate curated series exactly; find them before you concatenate the two files). Treat as exploratory material of uneven quality |

**Universe.** 50 assets in four asset classes — A (26 assets), B (8), C (9) and D (7) — across 12 countries. Assets in classes B, C and D each belong to one country. **Assets in class A have no natural country**; in the files they are assigned `country_7` by convention (a country that also has one class-C and one class-D asset). If you use country-level macro for class A, the convention gives them country 7's variables, which is one defensible choice among several (see step 2 of the workflow); in a widened-panel design you can give them whatever cross-country information you find sensible.

**Variables.** `x1` to `x5` are cross-sectional asset characteristics: each is computed for every asset every month from that asset's own prices and fundamentals, so at any date it varies across assets and can be used to rank them (one of the five is a risk measure). `x6` to `x164` are macroeconomic and financial-condition variables: rates, inflation, activity, external balances, credit, valuation, uncertainty and risk ratings, and some pre-computed transforms of them. Working out what a variable does — from its persistence, its scale, its cross-sectional behaviour and its relation to returns — is part of the project.

**Read this before modelling.**
- **Lag conventions.** `x2` and `x5` are *already* lagged one month: use them as they are to predict `excess_return` in the same row. `x1`, `x3`, `x4` and **every macro variable** are stored *contemporaneously*: shift them by one month within asset (or country) before using them as predictors. Getting this wrong silently introduces look-ahead. Macro data is also released with a delay; a one-month lag is the minimum, and you may argue for more.
- **Missing values.** The asset panel and the global file have no gaps, but a few leading months of `x2`, `x3` and `x5` in 2000–2002 were filled with the first observed value for a handful of assets; returns were never filled. In the curated country file, `x11` stops being updated before the end of the sample for three countries. In the extended file, seven columns have gaps. If you want to use a series with gaps, you have to deal with them yourself (see step 2 of the workflow).
- **You are free to bring in more data.** If you think other macro series, other asset characteristics, or other information would help the prediction, find it, document its source and its release timing, and treat it as part of your feature engineering. Anything you add is subject to the same lag rules as the data we provide.
- **Scales differ enormously by class.** Monthly return volatility ranges from under 1% to above 17% across assets, and its typical level differs systematically across the four classes. Think about this before pooling assets in one regression or one equal-weighted portfolio.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

_DATA_DIR = '/classes/41210_MiF_fall2026/Data/'
# _DATA_DIR = './'    # if the data files sit next to this notebook

panel = pd.read_csv(_DATA_DIR + 'asset_panel.csv', parse_dates=['date'])
info  = pd.read_csv(_DATA_DIR + 'asset_info.csv', index_col=0)
mac_g = pd.read_csv(_DATA_DIR + 'macro_global.csv', parse_dates=['date'])
mac_c = pd.read_csv(_DATA_DIR + 'macro_country.csv', parse_dates=['date'])
mac_x = pd.read_csv(_DATA_DIR + 'macro_country_extended.csv', parse_dates=['date'])
print(panel.shape, panel.date.min().date(), panel.date.max().date())
panel.head()


## Two rules that apply throughout

1. **Benchmarks before models.** Before you fit anything, write down the simple rules your model must beat and compute their out-of-sample performance. For return forecasts the benchmark is the trailing mean (or zero). For portfolios it is equal weight, risk parity ($w_i \propto 1/\hat\sigma_i$, with $\hat\sigma_i$ a trailing volatility computed from past returns) and time-series momentum ($w_i \propto \mathrm{sign}(\text{trailing 12-month return}_i)/\hat\sigma_i$), all with unit gross exposure and monthly rebalancing. All three can be built from `excess_return` alone.
2. **Time-ordered evaluation.** Fix the out-of-sample window and the initial training block in advance, before you fit anything, and say why you chose them. Every predictor must be known at the end of month $t-1$; every scaler, penalty and hyper-parameter is chosen on training data only; the trailing-mean benchmark for month $t$ is the mean of returns through month $t-1$ only, updated each month (the expanding-window harness of HW5 Problem 1.1, where the $R^2$ benchmark is the mean of the data seen so far).


## A suggested workflow

You do not have to follow this order, but a complete project touches each step.

**1. Know your data.** Cumulative excess returns by asset class; a table of annualized mean, volatility, Sharpe ratio and worst month per asset; the correlation matrix ordered by class (how strong is the block structure, and what does it imply for pooling?); rolling 12-month Sharpe by class (how persistent is performance?); the distribution, scale and persistence of each characteristic within each class, and a first look at its predictive content (average next-month return of assets sorted into terciles on it each month). Form a hypothesis about what each of `x1` to `x5` measures; you will need it to interpret your results.

**2. Feature engineering.** This is where most of the design lives.
- Apply the lag rule. Standardize characteristics cross-sectionally within month (across all assets or within class; z-score or rank) and say why.
- **If you use a macro series with missing values, deal with them first and write down what you did.** Diagnose the pattern for each series and country: does it stop early, or have holes in the middle? Then choose a treatment and justify that it uses *only information available at the time*. Forward-filling the last observed value and dropping the series are legitimate; filling backward, interpolating across a gap, or imputing with a full-sample mean all use the future. One trap worth your attention: a series that stops updating during a regime change, where carrying a stale value forward is not a small error. Check whether the series can be rebuilt, exactly or approximately, from other complete columns in either file, and report how close the rebuild is.
- **Add your own data if you have a hypothesis for it.** More macro series, other characteristics, anything you can source and date properly. Say where it came from, when each value became known, and lag it accordingly.
- Standardize global macro with *trailing* information only (a rolling z-score, for instance), never with the full sample.
- Map country macro to assets through `country`. Classes B, C and D each have their own country, so the mapping is direct; a *differential* against a base country (country 7 is the natural base) is often more informative than a level. Class A has no natural country. The files assign it country 7 by convention, and using country 7's macro for class A is one defensible choice; a cross-country average, cross-country dispersion, or no country macro at all are others. Say which you use and why.
- Consider cross-sectional macro transforms: the rank of a country's value across the 12 countries, its deviation from the cross-country mean, or a widened panel in which each asset sees its own country's macro *and* the cross-country average or every country's value. In the widened panel, class A can be given any cross-country information you find sensible rather than one country's.
- Consider interactions between characteristics and macro states, class dummies, and the extended file if you have a hypothesis for it.
- Choose the target. We recommend the **vol-scaled** excess return $y_{i,t} = r_{i,t}/\hat\sigma_{i,t-1}$ with $\hat\sigma$ a trailing volatility; explain why, and how you turn forecasts back into positions.
- End with one table listing every predictor and the count.

**3. Models and evaluation schemes.** Fit whatever models you find promising — OLS, ridge, lasso, random forests, boosted trees, neural networks, or any other method — and evaluate them under one or more schemes: static (fit once on the initial training block), expanding window (refit periodically on all data to date, the expanding-window scheme of Lecture 5 that you implemented in HW5 Problem 1), rolling window (refit on a fixed-length trailing window). Report $R^2_{OOS}$ against the trailing mean and against zero, model × scheme, and by asset class for the models you care about. Questions worth answering along the way: which scheme wins and why; which predictors carry the signal and whether that is stable across windows; whether macro adds anything once you refit without it; whether a **placebo** (every macro series circularly shifted by 36 months, real data at the wrong time) does about as well as the real macro (a persistent series stays partly correlated with a shifted copy of itself, so try more than one shift); whether per-class models beat the pooled one.

**4. From forecasts to portfolios.** Form portfolios from your forecasts in whatever way you find sensible (positions proportional to the forecast scaled by volatility, the sign of the forecast, long the top and short the bottom of a sort, or your own rule). State clearly how the portfolio is formed and rebalanced, and compare its performance with the benchmarks — Sharpe ratio at a minimum, and whatever else helps the reader judge it (return, volatility, drawdown, turnover, performance net of transaction costs, cumulative-return plots, sub-periods). Where you find outperformance, ask how much of it is exposure to the benchmark rules and how much is timing.


## What to submit

Two things, in this notebook:

1. **The analysis**, running top to bottom on the class server (or with `_DATA_DIR` pointed at a local copy of the data files), with every figure and table the write-up refers to.

2. **A write-up in the form of a research paper**, in markdown at the end of the notebook. The point of this project is to practise doing research, not to answer questions, so write it the way an empirical finance paper is written:

   - **Introduction.** The question, why it matters to someone who allocates money across countries and asset classes, what the literature on cross-sectional return predictability leads you to expect, and a one-paragraph preview of what you find.
   - **Data.** What the panel contains, how you treated it (missing values, lags, standardization, the country-to-asset mapping for macro), summary statistics and the plots that convey the structure of the data.
   - **Methodology.** The benchmarks and why they are the right ones; the models; the evaluation schemes and the out-of-sample window; how penalties and hyper-parameters were chosen; how forecasts become positions; how performance is measured. A reader should be able to reproduce your design from this section alone.
   - **Results.** Tables and figures with the numbers, presented in the order of the questions: forecastability of returns, the contribution of macro, and the portfolio evidence against the benchmarks.
   - **Interpretation and discussion.** What the results mean economically. Why the numbers come out the way they do; where the evidence is strong and where it is fragile; what you tried that did not work; what you would do next with more data or more time.
   - **Conclusion.** A summary a portfolio manager could read on their own.

   There is no length requirement in either direction. Length should follow from what you have to say; a clear negative result written up carefully is a good paper.


In [ ]:
# Your project starts here.


## Write-up

